# 02 — Causal inference: to what extent does treatment cause outcome?

The first two notebooks prepared the data and explored observed associations.

Now we answer the causal question:

> **To what extent does applying the backdoor defense, instead of baseline random filtering, cause the probability of successful backdoor detection to change?**

Domain interpretation:

> **To what extent does applying the backdoor defense, instead of random filtering, cause the probability of successful backdoor detection to change?**

### Treatment and outcome

| Variable | Meaning |
|---|---|
| `treatment = 0` | random filtering baseline |
| `treatment = 1` | backdoor defense applied |
| `outcome = 0` | detection failed |
| `outcome = 1` | detection succeeded |

### Tutorial path

00 Data preparation → 01 Correlational analysis → **02 Causal inference**


## 1. The causal estimand

For each unit, imagine two potential outcomes:

- **Y(1):** whether detection would succeed if the backdoor defense were applied;
- **Y(0):** whether detection would succeed under random filtering.

We cannot observe both potential outcomes for the same unit. The tutorial targets the **Average Treatment Effect (ATE)**:

**ATE = E[Y(1) − Y(0)]**

Because the outcome is binary, the ATE is a difference in detection-success probability.

For example, an ATE of `0.10` means that applying the backdoor defense causes an estimated **10 percentage-point increase in detection success rate**, on average, relative to random filtering.


## 2. Configure the causal analysis

The worksheet from notebook 01 provides statistical clues and causal notes. The hidden ground-truth files are used only after you have frozen your proposed DAG.

The graph uses Seaborn's `colorblind` categorical palette so causal roles are easier to distinguish.


In [ ]:
from pathlib import Path
import json

import numpy as np
import pandas as pd
from IPython.display import display
from dowhy import CausalModel

from src.causal_graph_ui import CausalGraphBuilder
from src.causal_tutorial_utils import (
    check_dowhy_version,
    load_analysis_data,
)

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

def default_params():
    return {
        "causal_dataset": "data/causal_data.csv",
        "dag_worksheet_path": "data/dag_worksheet.csv",
        "ground_truth_dag": "data/synthetic_ground_truth_edges.csv",
        "study_metadata": "data/synthetic_study_metadata.json",
        "treatment_column": "treatment",
        "outcome_column": "outcome",
        "covariate_columns": ['code_number_tokens', 'code_complexity', 'code_num_identifiers', 'code_num_strings', 'reviewer_experience', 'rollout_eligibility', 'noise_feature', 'inspection_intensity', 'manual_review_flag'],
        "graph_palette": "colorblind",
        "graph_edge_opacity": 0.35,
        "refuter_simulations": 50,
    }

params = default_params()

print("DoWhy version:", check_dowhy_version("0.14"))
params


## 3. Load the observed data

The hidden generating DAG is still not used here.

The model sees only the observed treatment, outcome, and measured variables.


In [ ]:
(
    analysis_df,
    treatment,
    outcome,
    covariates,
    excluded_covariates,
) = load_analysis_data(params)

print(f"Rows: {len(analysis_df):,}")
print(f"Backdoor-defense prevalence: {analysis_df[treatment].mean():.3f}")
print(f"Observed detection success rate: {analysis_df[outcome].mean():.3f}")
print(f"Covariates available for the graph: {len(covariates)}")
print(f"Excluded constant covariates: {excluded_covariates or 'none'}")

analysis_df.head()


## 4. Review the DAG worksheet from notebook 01

The worksheet records:

- what the data showed;
- when variables were measured;
- and any causal hypotheses you entered.

The worksheet is not itself a causal model. The DAG is where you make the causal assumptions explicit.


In [ ]:
worksheet_path = Path(params["dag_worksheet_path"])

if worksheet_path.exists():
    dag_worksheet = pd.read_csv(worksheet_path)
    display(dag_worksheet)
else:
    dag_worksheet = None
    print(
        "DAG worksheet not found. Run "
        "01_correlational_analysis.ipynb first."
    )


## 5. Build the DAG

The graph begins with the causal relationship we want to study:

`treatment → outcome`

That arrow represents the hypothesis that applying the backdoor defense can change detection success.

Now consider the other variables. Add an edge only when you can explain a plausible causal mechanism and temporal order.

Remember:

- a pre-treatment variable associated with both treatment and outcome may be a common cause;
- a post-treatment variable may lie on the causal pathway;
- a variable caused by two other variables may be a collider;
- a variable can strongly predict treatment without directly causing outcome.

The graph editor labels structural roles **after** you draw the graph. It does not infer causal arrows from correlation.


In [ ]:
graph_builder = CausalGraphBuilder(
    data_columns=analysis_df.columns,
    treatment=treatment,
    outcome=outcome,
    covariates=covariates,
    palette=params["graph_palette"],
    edge_opacity=params["graph_edge_opacity"],
)

graph_builder.display()


### Freeze your proposed DAG

When the graph represents the assumptions you are willing to defend, click **Use this DAG for analysis** and run the next cell.

Do this before revealing the hidden synthetic ground truth.


In [ ]:
analysis_dag = graph_builder.get_frozen_graph()

print(
    f"Using DAG with {analysis_dag.number_of_nodes()} nodes "
    f"and {analysis_dag.number_of_edges()} edges."
)


## 6. Create the DoWhy causal model

DoWhy now combines:

- the observed data;
- the treatment;
- the outcome;
- and the DAG you supplied.

The DAG—not the correlation matrix—determines which causal assumptions DoWhy uses.


In [ ]:
causal_model = CausalModel(
    data=analysis_df,
    treatment=treatment,
    outcome=outcome,
    graph=analysis_dag,
)

print("DoWhy causal model created from the frozen DAG.")


## 7. Identify the causal effect

Identification asks:

> **If this DAG is correct, can the ATE be written using quantities that are observable in the data?**

This step determines **what** should be estimated before choosing **how** to estimate it.


In [ ]:
identified_estimand = causal_model.identify_effect(
    proceed_when_unidentifiable=False
)

print(identified_estimand)


## 8. Estimate the ATE

We use inverse propensity-score weighting for the identified backdoor estimand.

Interpret the sign and magnitude in the original backdoor-defense context:

- positive ATE: the defense increases detection success on average;
- negative ATE: the defense decreases detection success on average;
- value near 0: little average causal change under the model assumptions.


In [ ]:
estimate = causal_model.estimate_effect(
    identified_estimand,
    method_name="backdoor.propensity_score_weighting",
    target_units="ate",
    control_value=0,
    treatment_value=1,
    method_params={
        "min_ps_score": 0.05,
        "max_ps_score": 0.95,
        "weighting_scheme": "ips_weight",
    },
)

estimated_ate = float(estimate.value)

print(estimate)
print(f"\nEstimated ATE: {estimated_ate:.4f}")
print(
    "Interpretation: the model estimates a "
    f"{estimated_ate * 100:.1f} percentage-point average change "
    "in detection success when using the backdoor defense rather than "
    "random filtering."
)


## 9. Refute the estimate

Refuters test whether the estimate behaves sensibly under selected perturbations.

They are useful diagnostics, but they do **not** prove that the DAG is correct or that all confounding has been removed.

### Placebo treatment

Treatment is randomly permuted. This breaks the original treatment assignment, so the placebo effect should generally be near zero.


In [ ]:
placebo_refutation = causal_model.refute_estimate(
    identified_estimand,
    estimate,
    method_name="placebo_treatment_refuter",
    placebo_type="permute",
    num_simulations=params["refuter_simulations"],
    random_seed=RANDOM_SEED,
)

print(placebo_refutation)


### Random common cause

An irrelevant random variable is added. A stable estimate should not change substantially simply because irrelevant information was added.


In [ ]:
random_common_cause_refutation = causal_model.refute_estimate(
    identified_estimand,
    estimate,
    method_name="random_common_cause",
    num_simulations=params["refuter_simulations"],
    random_seed=RANDOM_SEED,
)

print(random_common_cause_refutation)


# Reveal: compare your assumptions with the synthetic ground truth

Only now do we use the information that was hidden during graph construction.

Compare:

1. the DAG you proposed;
2. the DAG that generated the synthetic study;
3. your estimated ATE;
4. the known synthetic ATE.

The goal is not merely to count correct edges. Ask **why** the correlational evidence did or did not reveal each causal role.


In [ ]:
ground_truth = pd.read_csv(params["ground_truth_dag"])
display(ground_truth)

true_edges = set(zip(ground_truth["source"], ground_truth["target"]))
proposed_edges = set(analysis_dag.edges())

missing_edges = sorted(true_edges - proposed_edges)
extra_edges = sorted(proposed_edges - true_edges)

print("\nEdges in the synthetic DAG but missing from your DAG:")
print(missing_edges or "none")

print("\nEdges in your DAG but not in the synthetic DAG:")
print(extra_edges or "none")

metadata_path = Path(params["study_metadata"])
if metadata_path.exists():
    study_metadata = json.loads(metadata_path.read_text())
    true_ate = float(study_metadata["true_average_treatment_effect"])
    print(f"\nKnown synthetic ATE: {true_ate:.4f}")
    print(f"Estimated ATE:       {estimated_ate:.4f}")
    print(f"Estimation error:    {estimated_ate - true_ate:+.4f}")
else:
    print("\nSynthetic study metadata not found; rerun notebook 00.")


## Final interpretation

The three notebooks answer three different questions:

**Notebook 00 — What was observed?**  
Create one unit-level record with one treatment and one outcome.

**Notebook 01 — What patterns are visible?**  
Measure association and imbalance without calling them causal effects.

**Notebook 02 — To what extent does applying the backdoor defense, instead of baseline random filtering, cause the probability of successful backdoor detection to change?**  
State a DAG, identify an estimand, estimate the ATE, and test selected forms of robustness.

The main lesson is that choosing adjustment variables is a **causal reasoning problem**, not a correlation-ranking problem.
